# UN Sentiment & Tone Analysis — Production Processing

Production-oriented pipeline for the UN briefing sentiment and tone analysis.

**Features**
- Uses the existing `data/briefings.db` source database.
- Creates `analysis_run` and `sentiment_analysis` tables if they do not exist.
- Records model, Ollama package version, prompt version, prompt, temperature, and timestamps.
- Fixes the JSON prompt formatting bug in the previous notebook.
- Uses Ollama JSON output.
- Limits generated output to reduce unnecessary inference.
- Keeps the model loaded between requests.
- Uses bounded concurrency.
- Saves results incrementally to SQLite.
- Resumes safely after interruption.
- Exports results to CSV.

**Recommended:** test one section first, then a small sample, then run the full dataset.

## 0. Configuration

In [1]:
from pathlib import Path
import sqlite3
import json
import re
import time
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed
from importlib.metadata import version as package_version, PackageNotFoundError

import pandas as pd
from tqdm.auto import tqdm
import ollama

PROJECT_ROOT = Path(r"E:\correlaid-mentoring-project")
DB_PATH = PROJECT_ROOT / "data" / "briefings.db"
OUTPUT_CSV = PROJECT_ROOT / "data" / "sentiment_tone_analysis.csv"

MODEL_NAME = "llama3.2"
TEMPERATURE = 0
NUM_PREDICT = 50
KEEP_ALIVE = "10m"

# Start with 2. Increase only after benchmarking.
MAX_WORKERS = 2

# Results are committed after each batch.
COMMIT_BATCH_SIZE = 20

# None creates a new analysis run.
# To resume an interrupted run, set this to the existing analysis_id.
RESUME_ANALYSIS_ID = None

ANALYSIS_NAME = "UN briefing sentiment and tone analysis"
PROMPT_VERSION = "v1.1-production"

SENTIMENT_CATEGORIES = {"Positive", "Neutral", "Negative"}
TONE_CATEGORIES = {
    "Diplomatic",
    "Concerned",
    "Critical",
    "Urgent",
    "Optimistic",
    "Neutral/Informational",
}

if not DB_PATH.exists():
    raise FileNotFoundError(f"Database not found: {DB_PATH}")

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_ROOT)
print("Database:", DB_PATH)
print("Model:", MODEL_NAME)
print("Workers:", MAX_WORKERS)


C:\Users\Lenovo#\.virtualenvs\correlaid-mentoring-project-NQMA7taw\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project: E:\correlaid-mentoring-project
Database: E:\correlaid-mentoring-project\data\briefings.db
Model: llama3.2
Workers: 2


## 1. Analysis prompt

In [2]:
# IMPORTANT:
# Do not use ANALYSIS_PROMPT.format(text=text).
# The JSON braces would be interpreted as Python format fields.
# Use the unique placeholder and .replace() instead.

ANALYSIS_PROMPT = """
Analyze the following UN briefing section.

Classify its sentiment as exactly one of:
- Positive
- Neutral
- Negative

Classify its primary tone as exactly one of:
- Diplomatic
- Concerned
- Critical
- Urgent
- Optimistic
- Neutral/Informational

Return only a JSON object with exactly this structure:
{"sentiment": "<sentiment>", "tone": "<tone>"}

Do not provide explanations or additional fields.

Text:
__SECTION_TEXT__
""".strip()

print(ANALYSIS_PROMPT)


Analyze the following UN briefing section.

Classify its sentiment as exactly one of:
- Positive
- Neutral
- Negative

Classify its primary tone as exactly one of:
- Diplomatic
- Concerned
- Critical
- Urgent
- Optimistic
- Neutral/Informational

Return only a JSON object with exactly this structure:
{"sentiment": "<sentiment>", "tone": "<tone>"}

Do not provide explanations or additional fields.

Text:
__SECTION_TEXT__


## 2. Create / verify analysis database tables

In [3]:

conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON")

conn.execute("""
CREATE TABLE IF NOT EXISTS analysis_run (
    analysis_id INTEGER PRIMARY KEY AUTOINCREMENT,
    analysis_name TEXT NOT NULL,
    model_name TEXT NOT NULL,
    model_version TEXT,
    ollama_version TEXT,
    prompt_version TEXT NOT NULL,
    prompt TEXT NOT NULL,
    temperature REAL NOT NULL,
    created_at TEXT NOT NULL
)
""")

conn.execute("""
CREATE TABLE IF NOT EXISTS sentiment_analysis (
    analysis_result_id INTEGER PRIMARY KEY AUTOINCREMENT,
    analysis_id INTEGER NOT NULL,
    briefing_id INTEGER NOT NULL,
    section_id INTEGER NOT NULL,
    briefing_date TEXT,
    sentiment TEXT,
    tone TEXT,
    status TEXT NOT NULL,
    error_message TEXT,
    created_at TEXT NOT NULL,
    FOREIGN KEY (analysis_id) REFERENCES analysis_run(analysis_id),
    UNIQUE (analysis_id, briefing_id, section_id)
)
""")

conn.execute("""
CREATE INDEX IF NOT EXISTS idx_sentiment_analysis_analysis_id
ON sentiment_analysis(analysis_id)
""")

conn.execute("""
CREATE INDEX IF NOT EXISTS idx_sentiment_analysis_briefing_date
ON sentiment_analysis(briefing_date)
""")

conn.commit()

print("Analysis database tables are ready.")


Analysis database tables are ready.


## 3. Record Ollama/model metadata

In [4]:
def get_ollama_package_version():
    try:
        return package_version("ollama")
    except PackageNotFoundError:
        return "unknown"

def get_model_version():
    try:
        info = ollama.show(model=MODEL_NAME)

        if isinstance(info, dict):
            digest = info.get("digest")
            if digest:
                return str(digest)

            details = info.get("details", {})
            if isinstance(details, dict) and details.get("digest"):
                return str(details["digest"])

        return "available-from-ollama-show"

    except Exception as exc:
        print("Could not retrieve model metadata:", exc)
        return "unavailable"

MODEL_VERSION = get_model_version()
OLLAMA_VERSION = get_ollama_package_version()

print("Model name:", MODEL_NAME)
print("Model version/digest:", MODEL_VERSION)
print("Python ollama package:", OLLAMA_VERSION)


Model name: llama3.2
Model version/digest: available-from-ollama-show
Python ollama package: 0.6.2


## 4. Load valid UN briefing sections

In [5]:
query = """
SELECT
    s.section_id,
    s.briefing_id,
    b.briefing_date,
    s.section_order,
    s.title,
    s.text AS section_text
FROM section AS s
JOIN briefing AS b
    ON s.briefing_id = b.briefing_id
WHERE s.text IS NOT NULL
  AND TRIM(s.text) != ''
ORDER BY b.briefing_date, s.section_order
"""

df = pd.read_sql_query(query, conn)

df["briefing_date"] = pd.to_datetime(
    df["briefing_date"], errors="coerce"
)

duplicate_count = df.duplicated(
    ["briefing_id", "section_id"],
    keep=False
).sum()

df = (
    df.drop_duplicates(
        ["briefing_id", "section_id"],
        keep="first"
    )
    .reset_index(drop=True)
)

print("Rows loaded:", len(df))
print("Duplicate records found:", duplicate_count)
print("Rows after duplicate removal:", len(df))


Rows loaded: 2970
Duplicate records found: 0
Rows after duplicate removal: 2970


## 5. Create or resume an analysis run

In [6]:
def utc_now():
    return datetime.now(timezone.utc).isoformat()

if RESUME_ANALYSIS_ID is None:

    cur = conn.execute(
        """
        INSERT INTO analysis_run (
            analysis_name,
            model_name,
            model_version,
            ollama_version,
            prompt_version,
            prompt,
            temperature,
            created_at
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        """,
        (
            ANALYSIS_NAME,
            MODEL_NAME,
            MODEL_VERSION,
            OLLAMA_VERSION,
            PROMPT_VERSION,
            ANALYSIS_PROMPT,
            TEMPERATURE,
            utc_now(),
        ),
    )

    analysis_id = cur.lastrowid
    conn.commit()

    print("Created new analysis_id:", analysis_id)

else:

    row = conn.execute(
        """
        SELECT
            analysis_id,
            analysis_name,
            model_name,
            model_version,
            ollama_version,
            prompt_version,
            temperature,
            created_at
        FROM analysis_run
        WHERE analysis_id = ?
        """,
        (RESUME_ANALYSIS_ID,),
    ).fetchone()

    if row is None:
        raise ValueError(
            f"Analysis ID {RESUME_ANALYSIS_ID} does not exist."
        )

    analysis_id = RESUME_ANALYSIS_ID

    print("Resuming analysis_id:", analysis_id)
    print("Stored metadata:", row)


Created new analysis_id: 11


## 6. Test Ollama and validate JSON

In [7]:
def build_prompt(text):
    return ANALYSIS_PROMPT.replace("__SECTION_TEXT__", text)

def parse_and_validate(raw):
    raw = raw.strip()

    raw = re.sub(
        r"^```(?:json)?\s*|\s*```$",
        "",
        raw,
        flags=re.IGNORECASE
    ).strip()

    result = json.loads(raw)

    if set(result.keys()) != {"sentiment", "tone"}:
        raise ValueError(
            f"Unexpected JSON fields: {list(result.keys())}"
        )

    if result["sentiment"] not in SENTIMENT_CATEGORIES:
        raise ValueError(
            f"Invalid sentiment: {result['sentiment']}"
        )

    if result["tone"] not in TONE_CATEGORIES:
        raise ValueError(
            f"Invalid tone: {result['tone']}"
        )

    return result

def analyze_section(text):
    try:
        response = ollama.chat(
            model=MODEL_NAME,
            messages=[
                {
                    "role": "user",
                    "content": build_prompt(text),
                }
            ],
            format="json",
            options={
                "temperature": TEMPERATURE,
                "num_predict": NUM_PREDICT,
            },
            keep_alive=KEEP_ALIVE,
        )

        result = parse_and_validate(
            response["message"]["content"]
        )

        return (
            result["sentiment"],
            result["tone"],
            "success",
            None,
        )

    except Exception as exc:
        return (
            None,
            None,
            "error",
            f"{type(exc).__name__}: {exc}",
        )

test_text = df.iloc[0]["section_text"]

sentiment, tone, status, error = analyze_section(test_text)

print("Test result")
print("=" * 60)
print("Sentiment:", sentiment)
print("Tone:", tone)
print("Status:", status)
print("Error:", error)

if status != "success":
    raise RuntimeError(f"Ollama test failed: {error}")


Test result
Sentiment: Positive
Tone: Neutral/Informational
Status: success
Error: None


## 7. Optional benchmark — 20 sections

In [8]:
BENCHMARK_SIZE = min(20, len(df))

benchmark_start = time.time()

for row in tqdm(
    df.head(BENCHMARK_SIZE).itertuples(index=False),
    total=BENCHMARK_SIZE,
    desc="Benchmark"
):
    analyze_section(row.section_text)

benchmark_elapsed = time.time() - benchmark_start

print(f"Benchmark sections: {BENCHMARK_SIZE}")
print(f"Total time: {benchmark_elapsed:.2f} seconds")
print(
    f"Average: "
    f"{benchmark_elapsed / BENCHMARK_SIZE:.2f} seconds/section"
)


Benchmark: 100%|██████████| 20/20 [03:26<00:00, 10.31s/it]

Benchmark sections: 20
Total time: 206.35 seconds
Average: 10.32 seconds/section


## 8. Production Section 3 — resumable + incremental + bounded concurrency

In [9]:
def process_row(row):
    """
    Ollama work happens in worker threads.
    SQLite writes remain in the main thread.
    """
    sentiment, tone, status, error = analyze_section(
        row.section_text
    )

    return {
        "analysis_id": analysis_id,
        "briefing_id": int(row.briefing_id),
        "section_id": int(row.section_id),
        "briefing_date": (
            row.briefing_date.isoformat()
            if pd.notna(row.briefing_date)
            else None
        ),
        "sentiment": sentiment,
        "tone": tone,
        "status": status,
        "error_message": error,
        "created_at": utc_now(),
    }


def get_completed_section_ids():
    rows = conn.execute(
        """
        SELECT section_id
        FROM sentiment_analysis
        WHERE analysis_id = ?
          AND status = 'success'
        """,
        (analysis_id,),
    ).fetchall()

    return {int(row[0]) for row in rows}


completed_ids = get_completed_section_ids()

pending_df = df[
    ~df["section_id"].astype(int).isin(completed_ids)
].copy()

print("Total source sections:", len(df))
print("Already successful:", len(completed_ids))
print("Remaining to analyse:", len(pending_df))


Total source sections: 2970
Already successful: 0
Remaining to analyse: 2970


In [ ]:
# -------------------------------------------------------------------
# FULL PRODUCTION RUN
# -------------------------------------------------------------------
# Results are committed after every batch.
#
# If interrupted:
# 1. Note the analysis_id.
# 2. Set RESUME_ANALYSIS_ID = that ID near the top.
# 3. Restart the notebook.
# 4. Successful rows will be skipped automatically.
#
# Previous error rows are retried.

if pending_df.empty:

    print("Nothing to process. All sections are already successful.")

else:

    rows = list(pending_df.itertuples(index=False))
    total_started = time.time()
    processed_since_start = 0

    with ThreadPoolExecutor(
        max_workers=MAX_WORKERS
    ) as executor:

        for batch_start in range(
            0,
            len(rows),
            COMMIT_BATCH_SIZE
        ):

            batch = rows[
                batch_start:
                batch_start + COMMIT_BATCH_SIZE
            ]

            futures = [
                executor.submit(process_row, row)
                for row in batch
            ]

            batch_results = []

            for future in tqdm(
                as_completed(futures),
                total=len(futures),
                desc="Analysing UN briefing sections",
                leave=False,
            ):
                try:
                    batch_results.append(
                        future.result()
                    )
                except Exception as exc:
                    print(
                        "Unexpected worker error:",
                        type(exc).__name__,
                        exc,
                    )

            for result in batch_results:

                conn.execute(
                    """
                    INSERT OR IGNORE INTO sentiment_analysis (
                        analysis_id,
                        briefing_id,
                        section_id,
                        briefing_date,
                        sentiment,
                        tone,
                        status,
                        error_message,
                        created_at
                    )
                    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
                    """,
                    (
                        result["analysis_id"],
                        result["briefing_id"],
                        result["section_id"],
                        result["briefing_date"],
                        result["sentiment"],
                        result["tone"],
                        result["status"],
                        result["error_message"],
                        result["created_at"],
                    ),
                )

            conn.commit()

            processed_since_start += len(batch_results)

            elapsed = time.time() - total_started
            rate = (
                processed_since_start / elapsed
                if elapsed > 0
                else 0
            )

            print(
                f"Saved {processed_since_start}/"
                f"{len(pending_df)} results | "
                f"{rate:.2f} sections/sec"
            )

    print("Production analysis run finished.")


Saved 20/2970 results | 0.01 sections/sec


Saved 40/2970 results | 0.02 sections/sec


Saved 60/2970 results | 0.02 sections/sec


Saved 80/2970 results | 0.03 sections/sec


Saved 100/2970 results | 0.03 sections/sec


Saved 120/2970 results | 0.04 sections/sec


... (20 outputs left out)

Saved 620/2970 results | 0.04 sections/sec


Saved 660/2970 results | 0.04 sections/sec


Saved 680/2970 results | 0.04 sections/sec


Saved 700/2970 results | 0.04 sections/sec


## 9. Validate results

In [ ]:
status_df = pd.read_sql_query(
    """
    SELECT
        status,
        COUNT(*) AS count
    FROM sentiment_analysis
    WHERE analysis_id = ?
    GROUP BY status
    ORDER BY status
    """,
    conn,
    params=(analysis_id,),
)

print(status_df)

result_count = conn.execute(
    """
    SELECT COUNT(*)
    FROM sentiment_analysis
    WHERE analysis_id = ?
    """,
    (analysis_id,),
).fetchone()[0]

success_count = conn.execute(
    """
    SELECT COUNT(*)
    FROM sentiment_analysis
    WHERE analysis_id = ?
      AND status = 'success'
    """,
    (analysis_id,),
).fetchone()[0]

error_count = conn.execute(
    """
    SELECT COUNT(*)
    FROM sentiment_analysis
    WHERE analysis_id = ?
      AND status = 'error'
    """,
    (analysis_id,),
).fetchone()[0]

print()
print("Total stored:", result_count)
print("Successful:", success_count)
print("Errors:", error_count)
print("Source sections:", len(df))


In [ ]:
sentiment_df = pd.read_sql_query(
    """
    SELECT sentiment, COUNT(*) AS count
    FROM sentiment_analysis
    WHERE analysis_id = ?
      AND status = 'success'
    GROUP BY sentiment
    ORDER BY count DESC
    """,
    conn,
    params=(analysis_id,),
)

tone_df = pd.read_sql_query(
    """
    SELECT tone, COUNT(*) AS count
    FROM sentiment_analysis
    WHERE analysis_id = ?
      AND status = 'success'
    GROUP BY tone
    ORDER BY count DESC
    """,
    conn,
    params=(analysis_id,),
)

print("Sentiment distribution")
display(sentiment_df)

print("\nTone distribution")
display(tone_df)


## 10. Export results to CSV

In [ ]:
export_query = """
SELECT
    sa.analysis_id,
    sa.briefing_id,
    sa.section_id,
    sa.briefing_date,
    s.section_order,
    s.title,
    sa.sentiment,
    sa.tone,
    sa.status,
    sa.error_message,
    sa.created_at
FROM sentiment_analysis AS sa
LEFT JOIN section AS s
    ON s.section_id = sa.section_id
WHERE sa.analysis_id = ?
ORDER BY sa.briefing_date, s.section_order
"""

results_df = pd.read_sql_query(
    export_query,
    conn,
    params=(analysis_id,),
)

results_df.to_csv(
    OUTPUT_CSV,
    index=False,
)

print("Rows exported:", len(results_df))
print("CSV:", OUTPUT_CSV)

display(results_df.head())


## 11. Reproducibility metadata

In [ ]:
metadata_df = pd.read_sql_query(
    """
    SELECT
        analysis_id,
        analysis_name,
        model_name,
        model_version,
        ollama_version,
        prompt_version,
        temperature,
        created_at
    FROM analysis_run
    WHERE analysis_id = ?
    """,
    conn,
    params=(analysis_id,),
)

display(metadata_df)


## 12. Close database connection

In [ ]:
conn.close()
print("Database connection closed.")
